# Appendix: Apple hardware, measured

> Notebook 103's decoder on the CPU, GPU and Neural Engine, with 8-, 6- and 4-bit weights, and XNNPACK for comparison.

Notebook 103 builds a Core ML decoder and shows it's faster. This measures how much, on what hardware, at what
quality, over ten sentences. It converts models and scores a minute of audio many times over, so `nbdev-test` skips
it.

In [ ]:
import os
import tempfile
import time
from collections import Counter

import coremltools as ct
import coremltools.optimize.coreml as cto
import librosa
import numpy as np
import pandas as pd
import torch
from coremltools.models.compute_plan import MLComputePlan

from text_to_speech.coreml import (
    BUCKETS,
    Body,
    CoreMLDecoder,
    HarmonicSource,
    compile_coreml,
    exportable_decoder,
    tile,
    to_coreml,
)
from text_to_speech.data import metadata
from text_to_speech.voice import FRAME, SR, analyse, decode

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.14.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1101 23:09:54.621000 6516 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1101 23:09:54.763000 6516 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprec

## The bar

The decoder adds noise on purpose, so two runs with different seeds already differ. That's the bar: anything we do
should keep its output about as close to Kokoro's as two seeds are to each other, measured between log mel
spectrograms, and keep UTMOS's predicted MOS. The test set is ten sentences, as in notebook 190.

In [ ]:
texts = ["Speed comes from the voice itself, not from stretching the audio afterwards."]
texts += [t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9]
utterances = [analyse(t) for t in texts]
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).eval()


def mel(a):
    return np.log(np.maximum(librosa.feature.melspectrogram(y=a, sr=SR, n_fft=1024, hop_length=256, n_mels=80), 1e-5))


def distance(a, b):
    n = min(len(a), len(b))
    return np.abs(mel(a[:n]) - mel(b[:n])).mean()


@torch.no_grad()
def mos(a):
    return utmos(torch.tensor(a, dtype=torch.float32)[None], SR).item()


def kokoro(seed):
    torch.manual_seed(seed)
    return [decode(u) for u in utterances]


ref = kokoro(0)
seconds = sum(map(len, ref)) / SR
floor = np.mean([distance(a, b) for a, b in zip(ref, kokoro(1))])
print(f"{seconds:.1f}s of audio; two seeds differ by {floor:.3f}; MOS {np.mean([mos(a) for a in ref]):.2f}")
results = []


@torch.no_grad()
def evaluate(name, decoder, repeats=2):
    "decoder(utterance) -> audio. Times it over the ten sentences and scores it against Kokoro."
    decoder(utterances[0])
    t = time.perf_counter()
    for _ in range(repeats):
        outs = [np.asarray(decoder(u), dtype=np.float32).squeeze() for u in utterances]
    elapsed = (time.perf_counter() - t) / repeats
    d = float(np.mean([distance(a, b) for a, b in zip(outs, ref)]))
    results.append(
        {
            "decoder": name,
            "x real time": float(seconds / elapsed),
            "vs seed floor": float(d / floor),
            "MOS": float(np.mean([mos(a) for a in outs])),
        }
    )
    print(" | ".join(f"{v:.2f}" if isinstance(v, float) else v for v in results[-1].values()))

61.8s of audio; two seeds differ by 0.070; MOS 4.52


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Using cache found in /Users/henrydashwood/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## Cheap wins on the CPU

Threads first. This M3 Pro has 6 performance cores, which is PyTorch's default.

In [ ]:
for n in [1, 6]:
    torch.set_num_threads(n)
    evaluate(f"PyTorch CPU, {n} thread{'s' if n > 1 else ''}", decode)

PyTorch CPU, 1 thread | 5.01 | 0.99 | 4.52
PyTorch CPU, 6 threads | 8.10 | 1.00 | 4.52


One core manages about 5x real time and six only about 9x: the decoder doesn't parallelise well on the CPU.
16-bit doesn't help either: PyTorch has no fast 16-bit convolutions on the CPU, and `torch.autocast` to bfloat16 ran
at 0.1x real time. `torch.compile` was slower than eager.

The exportable decoder from notebook 103, still in PyTorch:

In [ ]:
dec = exportable_decoder()
evaluate("PyTorch CPU, exportable", lambda u: dec(u.asr, u.F0, u.N, u.style).squeeze().numpy())

PyTorch CPU, exportable | 9.55 | 2.36 | 4.52


Without complex numbers, PyTorch's own GPU backend (MPS) runs it at about 25x real time. It isn't measured here
because using MPS and Core ML in one process crashed.

## Core ML

In [ ]:
for units in ["CPU_ONLY", "CPU_AND_GPU"]:
    evaluate(f"Core ML 16-bit, {units}", CoreMLDecoder(units))

Core ML 16-bit, CPU_ONLY | 19.93 | 4.81 | 4.49
Core ML 16-bit, CPU_AND_GPU | 29.87 | 3.91 | 4.52


The GPU through Core ML is the fastest so far, at full quality. The Neural Engine is missing because it won't take
inputs this long: asking for it fails to build at these sizes.

## The Neural Engine

It has limits on tensor sizes, and the decoder's last layers run on 120 time steps for every frame. Asking Core ML's
compute plan where each operation would run, at smaller sizes:

In [ ]:
source, body = HarmonicSource(dec).eval(), Body(dec).eval()


def plan(T):
    "Speed and where each op runs, for a body of T frames, on the CPU, GPU and Neural Engine."
    u = utterances[0]
    with torch.no_grad():
        args = (u.asr[..., :T], u.F0[:, : 2 * T], u.N[:, : 2 * T], u.style, source(u.F0[:, : 2 * T]))
        path = os.path.join(tempfile.mkdtemp(), "body.mlpackage")
        to_coreml(body, args).save(path)
    compiled = ct.utils.compile_model(path)
    feed = dict(zip(["asr", "f0", "n", "style", "har"], [a.numpy() for a in args]))
    for units in ["CPU_ONLY", "CPU_AND_GPU", "CPU_AND_NE"]:
        cp = MLComputePlan.load_from_path(compiled, compute_units=getattr(ct.ComputeUnit, units))
        ops = cp.model_structure.program.functions["main"].block.operations
        usage = [cp.get_compute_device_usage_for_mlprogram_operation(op) for op in ops]
        devices = Counter(type(u.preferred_compute_device).__name__[2:-13] for u in usage if u)
        m = ct.models.CompiledMLModel(compiled, compute_units=getattr(ct.ComputeUnit, units))
        m.predict(feed)
        t = time.perf_counter()
        for _ in range(10):
            m.predict(feed)
        elapsed = (time.perf_counter() - t) / 10
        print(f"{T:>4} frames, {units:<12} {T * 0.025 / elapsed:5.1f}x real time  {dict(devices)}")


for T in [64, 128]:
    plan(T)

  64 frames, CPU_ONLY      37.1x real time  {'CPU': 799}
  64 frames, CPU_AND_GPU   50.7x real time  {'CPU': 91, 'GPU': 708}
  64 frames, CPU_AND_NE    57.6x real time  {'NeuralEngine': 778, 'CPU': 21}
 128 frames, CPU_ONLY      37.3x real time  {'CPU': 799}
 128 frames, CPU_AND_GPU   50.1x real time  {'GPU': 799}
 128 frames, CPU_AND_NE     5.5x real time  {'CPU': 798}


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/kokoro/istftnet.py:380: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  out = (out + self._shortcut(x)) * torch.rsqrt(torch.tensor(2))
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 120.02 passes/s]


Up to 64 frames (1.6 s) nearly every operation runs on the Neural Engine, and it's the fastest of the three. At 128
the Neural Engine refuses and everything falls back to the CPU. So using it means decoding in short windows, with
overlaps cross-faded.

But the decoder normalises each channel over the whole time axis (instance norm, inside every AdaIN layer), so a
window sees its own statistics rather than the sentence's. That changes the sound:

In [ ]:
SAMPLES, HAR = FRAME, 120  # Audio samples and source columns per frame


@torch.no_grad()
def windowed(u, W=64, overlap=16):
    "Decode in windows of W frames, overlapping and cross-faded. The source is computed for the whole sentence."
    T = u.asr.shape[-1]
    har = source(u.F0)
    if T <= W:
        return body(u.asr, u.F0, u.N, u.style, har).squeeze()
    starts = list(range(0, T - W, W - overlap)) + [T - W]
    out, weight, ramp = torch.zeros(T * SAMPLES), torch.zeros(T * SAMPLES), torch.linspace(0, 1, overlap * SAMPLES)
    for k, a in enumerate(starts):
        cut = slice(a * SAMPLES, (a + W) * SAMPLES)
        y = body(
            u.asr[..., a : a + W],
            u.F0[:, 2 * a : 2 * (a + W)],
            u.N[:, 2 * a : 2 * (a + W)],
            u.style,
            har[..., a * HAR : (a + W) * HAR + 1],
        ).squeeze()
        w = torch.ones(W * SAMPLES)
        if k > 0:
            w[: len(ramp)] = ramp
        if k < len(starts) - 1:
            w[-len(ramp) :] = ramp.flip(0)
        out[cut] += y * w
        weight[cut] += w
    return out / weight


evaluate("PyTorch CPU, 64-frame windows", windowed)

PyTorch CPU, 64-frame windows | 5.56 | 8.29 | 4.42


A small but real drop, and the overlap length makes no difference (I tried 8 to 24 frames), so it's the statistics,
not the seams. 32-frame windows were worse again, about 3.95. For Kokoro, then, the GPU is the better choice: full
quality and nearly as fast. For a model of our own, it's a design rule: normalise without looking at the whole
sentence, and the Neural Engine is free to use.

## Smaller weights

The body is 107 MB at 16 bits. Two ways to shrink it: linear 8-bit quantisation, and palettisation, where each weight
becomes an index into a small table of values that k-means picks per layer, or per group of 16 channels.

Palettised models crash Core ML's Python bindings on the GPU with this version (a GIL error inside `predict`), so
they're measured on the CPU, where the 16-bit body is the comparison.

In [ ]:
def size_mb(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6


def compressed(compress, units):
    "A decoder like CoreMLDecoder, with the body's weights compressed by `compress`."
    folder, bodies = tempfile.mkdtemp(), {}
    for T in BUCKETS:
        with torch.no_grad():
            F0 = torch.zeros(1, 2 * T)
            args = (torch.zeros(1, 512, T), F0, torch.zeros(1, 2 * T), torch.zeros(1, 128), source(F0))
            bodies[T] = compile_coreml(compress(to_coreml(body, args)), f"{folder}/body{T}.mlpackage", units)
    print(f"body: {size_mb(f'{folder}/body256.mlpackage'):.0f} MB")

    @torch.no_grad()
    def decoder(u):
        T = u.asr.shape[-1]
        B = next(b for b in BUCKETS if b >= T)
        F0 = tile(u.F0, 2 * B)
        inputs = {"asr": tile(u.asr, B), "f0": F0, "n": tile(u.N, 2 * B), "style": u.style, "har": source(F0)}
        return bodies[B].predict({k: v.numpy() for k, v in inputs.items()})["out"].squeeze()[: T * FRAME]

    return decoder


def palette(nbits, **kw):
    config = cto.OptimizationConfig(global_config=cto.OpPalettizerConfig(nbits=nbits, mode="kmeans", **kw))
    return lambda m: cto.palettize_weights(m, config)


linear8 = lambda m: cto.linear_quantize_weights(
    m, cto.OptimizationConfig(global_config=cto.OpLinearQuantizerConfig(mode="linear_symmetric"))
)
for name, compress, units in [
    ("16-bit", lambda m: m, "CPU_ONLY"),
    ("8-bit linear", linear8, "CPU_AND_GPU"),
    ("6-bit palette", palette(6), "CPU_ONLY"),
    ("4-bit palette, per 16 channels", palette(4, granularity="per_grouped_channel", group_size=16), "CPU_ONLY"),
]:
    evaluate(f"Core ML {name}, {units}", compressed(compress, units))

body: 107 MB
Core ML 16-bit, CPU_ONLY | 24.08 | 4.80 | 4.49
body: 54 MB
Core ML 8-bit linear, CPU_AND_GPU | 16.09 | 3.94 | 4.51
body: 40 MB
Core ML 6-bit palette, CPU_ONLY | 22.55 | 4.95 | 4.49
body: 29 MB
Core ML 4-bit palette, per 16 channels, CPU_ONLY | 23.65 | 5.94 | 4.45


Running compression pass linear_quantize_weights: 100%|##########| 362/362 [00:00<00:00, 527.25 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass linear_quantize_weights: 100%|##########| 362/362 [00:00<00:00, 498.54 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass linear_quantize_weights: 100%|##########| 362/362 [00:00<00:00, 532.97 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass palettize_weights: 100%|##########| 362/362 [00:19<00:00, 18.58 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 94.10 passes/s]
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


8 bits costs nothing audible and 4 bits per group of channels costs a little. On the GPU, though, 8-bit was slower
than 16-bit: Core ML expands the weights as it goes.

## XNNPACK

For comparison, the CPU backend Android apps would use through ExecuTorch. ExecuTorch is an optional dependency,
because it brings torchao, which prints warnings on every import: install it with `uv sync --group xnnpack`. It needs constant weights, which is why
`exportable_decoder` folds weight normalisation: without that, it accelerated 5 of the 75 convolutions.

In [ ]:
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.devtools.backend_debug import get_delegation_info
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime

T, u = 128, utterances[0]
with torch.no_grad():
    args = tuple(
        a.contiguous() for a in (u.asr[..., :T], u.F0[:, : 2 * T], u.N[:, : 2 * T], u.style, source(u.F0[:, : 2 * T]))
    )
    edge = to_edge_transform_and_lower(torch.export.export(body, args), partitioner=[XnnpackPartitioner()])
print(get_delegation_info(edge.exported_program().graph_module).get_summary())
path = os.path.join(tempfile.mkdtemp(), "body.pte")
with open(path, "wb") as f:
    f.write(edge.to_executorch().buffer)
method = Runtime.get().load_program(path).load_method("forward")
method.execute(list(args))
t = time.perf_counter()
method.execute(list(args))
print(f"{os.path.getsize(path) / 1e6:.0f} MB, {T * 0.025 / (time.perf_counter() - t):.1f}x real time on 6 cores")

Total delegated subgraphs: 25
Number of delegated nodes: 845
Number of non-delegated nodes: 736

213 MB, 5.2x real time on 6 cores


[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:167] Number of efficient cores 6


That's slower than plain PyTorch on the CPU. Most of the convolutions run on XNNPACK, but over six hundred operations
around the instance norms don't, and XNNPACK only takes means over image-shaped tensors. Notebook 104 takes the GPU
route instead, through LiteRT, and gets about 10x real time on a Pixel 10a.

## Summary

In [ ]:
pd.DataFrame(results).round(2)

,decoder,x real time,vs seed floor,MOS
0,"PyTorch CPU, 1 thread",5.01,0.99,4.52
1,"PyTorch CPU, 6 threads",8.10,1.00,4.52
2,"PyTorch CPU, exportable",9.55,2.36,4.52
3,"Core ML 16-bit, CPU_ONLY",19.93,4.81,4.49
4,"Core ML 16-bit, CPU_AND_GPU",29.87,3.91,4.52
5,"PyTorch CPU, 64-frame windows",5.56,8.29,4.42
6,"Core ML 16-bit, CPU_ONLY",24.08,4.80,4.49
7,"Core ML 8-bit linear, CPU_AND_GPU",16.09,3.94,4.51
8,"Core ML 6-bit palette, CPU_ONLY",22.55,4.95,4.49
9,"Core ML 4-bit palette, per 16 channels, CPU_ONLY",23.65,5.94,4.45


- **The decoder is the whole job**: 90% of the time. The front half can stay on the CPU.
- **The GPU through Core ML is the winner on Apple hardware**: three to four times faster than PyTorch on the CPU, at
  full quality.
- **The Neural Engine is faster still per window, but only for windows of 1.6 s or less**, and windows cost quality
  because of instance norm.
- **Weights can shrink from 107 MB to about 29 MB** with 4-bit palettisation, for a small drop in MOS.
- **XNNPACK on the CPU is slower than PyTorch**: it leaves the instance norms on slow kernels.

Two lessons for the voice we train ourselves: avoid instance norm over the whole sentence, so it can run in windows
on the Neural Engine, and keep anything that accumulates over the sentence (like phase) out of the network.